## **Approach to LIDA Composting Project (Iteration 1)**

#### *Data preprocessing*
- Using the lida_tabletrimmed csv file, we'll gather all the feature sets and filter out the rows to be recorded **daily**. 
- All values will be cleaned and converted to positive values. 

#### *Data Oversampling*
- With the minuscule amount of data we have for training, an oversample technique must be implemented as compensation.
- We will implemented an oversampling technique that generates multiple synthetic data that preserves the time series qualities of composting

#### *Change Point Detection*
#### *Future Forecaster*


---

### **Data Init**

In [1]:
import pandas as pd
import numpy as np

file_path = "ideal_inflated_data.csv"
df = pd.read_csv(file_path)

### **Filtering rows by day**

In [3]:
# filtering out the rows by date. (daily)

df['timestamp'] = pd.to_datetime(df['timestamp'])
df_daily = df.groupby(df['timestamp'].dt.date).first().reset_index(drop=True)

In [4]:
df_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Day                  26 non-null     int64         
 1   temperature_active1  15 non-null     float64       
 2   temperature_active2  15 non-null     float64       
 3   temperature_active3  15 non-null     float64       
 4   temperature_active4  15 non-null     float64       
 5   temperature_curing1  11 non-null     float64       
 6   temperature_curing2  11 non-null     float64       
 7   moisture_active1     15 non-null     float64       
 8   moisture_active2     15 non-null     float64       
 9   moisture_curing1     11 non-null     float64       
 10  moisture_curing2     11 non-null     float64       
 11  oxygen               26 non-null     float64       
 12  co2                  26 non-null     float64       
 13  methane              26 non-null     

### **Dropping columns we don't need**

In [6]:
# # Dropping unnecessary cols
# df_daily.drop(columns=['device_id', 'lid', 'automation_active'], inplace=True)

In [7]:
numeric_cols = df_daily.select_dtypes(include=[np.number]).columns
df_daily[numeric_cols] = df_daily[numeric_cols].abs()

In [8]:
df_daily.head(21)

,Day,temperature_active1,temperature_active2,temperature_active3,temperature_active4,temperature_curing1,temperature_curing2,moisture_active1,moisture_active2,moisture_curing1,moisture_curing2,oxygen,co2,methane,run_id,timestamp
0,0,35.267501,36.316367,34.668588,35.788367,NaN,NaN,57.971782,60.067442,NaN,NaN,0.210000,0.005000,0.003000,1,2025-01-01
1,1,37.524641,38.409615,36.808272,37.752215,NaN,NaN,57.082293,58.722955,NaN,NaN,0.193767,0.023053,0.003000,1,2025-01-02
2,2,39.986224,40.409268,38.836486,39.909030,NaN,NaN,56.066820,57.375116,NaN,NaN,0.141459,0.046475,0.003000,1,2025-01-03
3,3,41.904339,42.467011,40.950690,42.326743,NaN,NaN,54.898553,56.628448,NaN,NaN,0.134213,0.072468,0.003000,1,2025-01-04
4,4,47.803883,48.150497,46.651595,48.246501,NaN,NaN,53.004872,55.957362,NaN,NaN,0.178757,0.100284,0.003000,1,2025-01-05
5,5,53.492064,53.734845,52.283863,53.633232,NaN,NaN,52.284912,54.246294,NaN,NaN,0.210000,0.129542,0.003000,1,2025-01-06
6,6,59.008203,59.976363,58.376753,59.305930,NaN,NaN,50.839361,52.981348,NaN,NaN,0.210000,0.130000,0.003000,1,2025-01-07
7,7,57.236117,57.299697,56.523561,57.372992,NaN,NaN,49.829142,52.344201,NaN,NaN,0.210000,0.125419,0.003000,1,2025-01-08
8,8,55.061425,55.650841,54.216660,55.227313,NaN,NaN,48.957636,50.800162,NaN,NaN,0.181755,0.120974,0.003000,1,2025-01-09
9,9,53.055280,53.751490,52.208590,53.365067,NaN,NaN,47.678303,49.645873,NaN,NaN,0.130473,0.146659,0.003000,1,2025-01-10


### **Oversampling implementation**
1. Interpolate the Base Data
2. Generate Multiple Synthetic Runs
3. Add Controlled Random Variation
4. Keep Time Continuous across runs
5. Label and combine

In [9]:
import pandas as pd
import numpy as np

def generate_synthetic_compost(df, time_col='time_stamp', freq='D', noise_factor=0.02, n_runs=10):
    """
    Generate synthetic composting runs from a small dataset,
    continuous in time and without NaNs, preserving compost cycle shape.
    """
    
    # Columns that should not be noised
    non_numeric_cols = [time_col, 'device_id', 'automation_active', 'lid']
    
    # Ensure datetime
    df[time_col] = pd.to_datetime(df[time_col])
    
    # Sort by time and set index for resampling
    df = df.sort_values(time_col).set_index(time_col)

    # Interpolate to higher resolution
    df_interp = df.resample(freq).interpolate(method='linear')
    
    synthetic_runs = []
    current_start = df_interp.index.min()
    
    for run_id in range(n_runs):
        df_aug = df_interp.copy()
        
        # Add noise to numeric columns only
        for col in df_aug.columns:
            if col not in non_numeric_cols and pd.api.types.is_numeric_dtype(df_aug[col]):
                noise = np.random.normal(
                    0, noise_factor * df_aug[col].std(), size=len(df_aug)
                )
                df_aug[col] += noise
        
        # Assign continuous timestamps for this run
        run_start_date = current_start
        df_aug.index = pd.date_range(
            start=run_start_date, 
            periods=len(df_aug), 
            freq=freq
        )
        
        # Drop NaNs (should be rare after interpolation)
        df_aug = df_aug.dropna()
        
        # Add run_id and reset index to restore time_col
        df_aug['run_id'] = run_id
        df_aug = df_aug.reset_index().rename(columns={'index': time_col})
        
        synthetic_runs.append(df_aug)
        
        # Move start date for next run
        current_start = df_aug[time_col].max() + pd.Timedelta(days=1)
    
    # Combine all runs
    return pd.concat(synthetic_runs, ignore_index=True)


In [12]:
synthetic_df = generate_synthetic_compost(df_daily, time_col='timestamp', freq='1D', noise_factor=0.03, n_runs=20)

In [13]:
synthetic_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220 entries, 0 to 219
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   timestamp            220 non-null    datetime64[ns]
 1   Day                  220 non-null    float64       
 2   temperature_active1  220 non-null    float64       
 3   temperature_active2  220 non-null    float64       
 4   temperature_active3  220 non-null    float64       
 5   temperature_active4  220 non-null    float64       
 6   temperature_curing1  220 non-null    float64       
 7   temperature_curing2  220 non-null    float64       
 8   moisture_active1     220 non-null    float64       
 9   moisture_active2     220 non-null    float64       
 10  moisture_curing1     220 non-null    float64       
 11  moisture_curing2     220 non-null    float64       
 12  oxygen               220 non-null    float64       
 13  co2                  220 non-null  

In [15]:
synthetic_df.head(50)

,timestamp,Day,temperature_active1,temperature_active2,temperature_active3,temperature_active4,temperature_curing1,temperature_curing2,moisture_active1,moisture_active2,moisture_curing1,moisture_curing2,oxygen,co2,methane,run_id
0,2025-01-16,15.140487,38.302224,38.622744,37.605111,38.555919,31.523292,32.429798,40.913738,42.329852,44.712094,45.481055,0.132563,0.123192,0.102496,0
1,2025-01-17,16.132191,38.390428,38.501780,37.173314,38.140160,29.498119,30.767096,40.369340,42.304013,43.129401,43.317219,0.151415,0.120422,0.028488,0
2,2025-01-18,17.100005,38.388526,38.550864,37.046124,38.466544,27.868951,29.044804,40.836239,42.351603,40.411565,40.817212,0.208754,0.114801,0.003102,0
3,2025-01-19,18.248859,38.148254,38.507717,37.199316,38.323355,26.216026,27.146346,40.949731,42.438990,38.986231,40.086746,0.208841,0.112743,0.003712,0
4,2025-01-20,19.474779,38.292603,38.478654,37.268787,38.164526,24.457872,25.032417,40.303949,42.609255,36.973265,38.470538,0.212276,0.109957,0.003260,0
5,2025-01-21,20.101053,38.189012,39.019875,37.515220,38.570233,22.449227,23.523513,40.895391,42.481270,35.256788,36.118684,0.177420,0.106792,0.002500,0
6,2025-01-22,20.968423,38.039631,38.555545,37.338453,38.230171,22.199537,22.955159,40.948557,42.469462,34.406189,34.754076,0.133602,0.103299,0.003446,0
7,2025-01-23,22.204337,37.883181,38.609173,37.401771,38.256936,21.732482,22.563657,41.062405,42.238118,33.003137,34.151316,0.141699,0.101061,0.003535,0
8,2025-01-24,23.308939,38.028391,38.267324,37.503724,38.470679,21.756252,22.625338,40.913001,42.653588,30.735144,31.625599,0.194151,0.098721,0.002699,0
9,2025-01-25,24.330795,38.193411,38.690068,36.931034,38.188797,21.766019,22.028502,40.695007,42.500094,29.873971,30.962534,0.209417,0.095662,0.002957,0


In [ ]:
# synthetic_df.to_csv("oversampled.csv")